In [1]:
import numpy as np
from abc import ABC, abstractmethod

In [2]:
class Layer(ABC):
    @abstractmethod
    def __init__(self):
        pass
    
    @abstractmethod
    def forward(self, x):
        pass
    
    @abstractmethod
    def backward(self, upstream_grad):
        pass

In [3]:
class MathematicalOperator:
    @abstractmethod
    def __init__(self):
        pass

    def forward(self, x):
        pass

    def backward(self, upstream_grad):
        pass

In [4]:
class GradTensor:
    def __init__(self, tensor):
        self.data = tensor
        self.shape = self.data.shape
        self.grad = None
    
    def _zero_grad(self):
        self.grad = None

In [5]:
class LinearLayer(Layer):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = GradTensor(np.random.rand(
            self.in_features,
            self.out_features
        ))

        self.bias = GradTensor(np.zeros(
            self.out_features
        ))
    
    def forward(self, x):
        self.x = x
        return np.matmul(self.x, self.weight.data) + self.bias.data
    
    def backward(self, upstream_grad):
        self.weight.grad = np.matmul(self.x.T, upstream_grad) # (I x O) --> (I x B) @ (B x O)
        self.bias.grad = np.sum(upstream_grad, axis = 0, keepdims=True)
        return np.matmul(upstream_grad, self.weight.grad.T)
    
    def __call__(self, x):
        return self.forward(x)

In [6]:
class NaiveSoftmax(MathematicalOperator):
    def __init__(self):
        pass

    def forward(self, x):
        adj_exp_x = np.exp(x - np.max(x, axis = -1, keepdims=True))
        self.probs = adj_exp_x / np.sum(adj_exp_x, keepdims=True, axis = -1)
        return self.probs
    
    def __call__(self, x):
        return self.forward(x)
    
    def backward(self, upstream_grad):
        """
        Computes del(Softmax)/del(input_to_softmax): each input to the softmax will tell us how much they effected the Softmax
        Therefore we can say that, shape of Derivative of Softmax w.r.t its input is self.x.shape
        """
        self.grad_tensor = np.zeros_like(self.probs)
        for i in range(self.probs.shape[0]):
            j = self.probs[i].reshape(-1, 1) * self.probs[i].reshape(1, -1)
            j[np.diag_indices(j.shape[0])] = self.probs[i] * (1 - self.probs[i])
            a = upstream_grad[i] @ j
            self.grad_tensor[i] = a
        return self.grad_tensor

In [7]:
class LayerNorm(Layer):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return super().forward(x)
    
    def backward(self, grad):
        return super().backward(grad) 

In [8]:
batch = 10
in_features = 3
n_classes = 2
x = np.random.rand(batch, in_features)
layer_1 = LinearLayer(in_features, n_classes)
logits = layer_1.forward(x)
layer_1.backward(np.random.rand(batch, n_classes))

array([[5.01392692, 3.41605065, 4.80076021],
       [1.42379999, 0.95511557, 1.35316066],
       [2.94176473, 2.06565155, 2.85823357],
       [2.91259776, 2.07601598, 2.85076486],
       [5.7199393 , 4.22116775, 5.69604811],
       [4.05659144, 2.84312608, 3.93779467],
       [5.1950968 , 3.59377436, 5.01096125],
       [4.37471373, 3.09143602, 4.26375219],
       [5.89394862, 4.30237223, 5.8373878 ],
       [5.5582949 , 3.96853386, 5.44485631]])

In [9]:
softmax = NaiveSoftmax()
probs = softmax(logits)
softmax.backward(np.random.rand(batch, n_classes))

array([[0.29428466, 0.29428466],
       [0.23157979, 0.23157979],
       [0.18917447, 0.18917447],
       [0.26058426, 0.26058426],
       [0.15045442, 0.15045442],
       [0.36486529, 0.36486529],
       [0.25720208, 0.25720208],
       [0.19458505, 0.19458505],
       [0.07873552, 0.07873552],
       [0.36505133, 0.36505133]])